### Imports

In [ ]:
# --- Imports ---
from supabase import create_client, Client
import time
from datetime import datetime
from dotenv import load_dotenv
import os
import pdfplumber
import tempfile
import os
import requests
from openai import OpenAI
from supabase import create_client, Client
import json
from fuzzywuzzy import process
import json
import re
import pandas as pd



### LLMHandler

In [ ]:
# Fixed LLMHandler class - copy this to replace the existing class in tester.ipynb

class LLMHandler: 
    def __init__(self, supabase_client: Client, organization_id: str, job_id: str, model: str = "deepseek-chat"):
        self.api_key = os.getenv("DEEPSEEK_API_KEY")
        if not self.api_key:
            raise RuntimeError("Please set DEEPSEEK_API_KEY environment variable")

        
        self.base_url = "https://api.deepseek.com"
        self.client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        self.organization_id = organization_id
        self.supabase_client = supabase_client
        self.job_id = job_id
        self.model = model

        # DeepSeek pricing (per 1M tokens, convert to per token)
        # https://platform.deepseek.com/api-docs/pricing/
        self.input_token_price = 0.28 / 1_000_000  # $0.14 per 1M tokens
        self.output_token_price = 0.42 / 1_000_000  # $0.28 per 1M tokens
        self.cache_hit_price = 0.028 / 1_000_000  # $0.014 per 1M cache hit tokens (90% discount)

    def call_deepseek_chat(self, prompt: str, model: str = "deepseek-chat", temperature: float = 0.0, max_tokens: int = 1000, use_json_mode: bool = False):
        """
        Call DeepSeek API with optional JSON mode.
        
        Args:
            prompt: The prompt to send to the model
            model: Model name (default: deepseek-chat)
            temperature: Temperature for sampling (default: 0.0)
            max_tokens: Maximum tokens to generate (default: 1000)
            use_json_mode: If True, use JSON response format (prompt must mention "json")
        """
        try:
            # Build API call parameters
            params = {
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": temperature,
                "max_tokens": max_tokens
            }
            
            # Only add response_format if JSON mode is requested
            # OpenAI requires the prompt to mention "json" when using json_object mode
            if use_json_mode:
                params["response_format"] = {"type": "json_object"}
            
            response = self.client.chat.completions.create(**params)
            
            self.log_token_usage(self.job_id, response)
            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"API call error: {e}")
            raise

    def classify_document(self, formatted_text: str, document_types: list[str]):
        """
        Classify a document into one of the predefined types.
        Returns the document type name or 'unknown' if no good match.
        """
        # Format document types as numbered list for clarity
        types_list = "\n".join([f"{i+1}. {dt}" for i, dt in enumerate(document_types)])
        
        prompt = f"""Document text:
            {formatted_text[:1000]}

            Instructions:
            Classify this document into ONE of these types:
            {types_list}

            Return ONLY the exact document type name, nothing else."""
        
        # Don't use JSON mode for classification - we want plain text response
        res = self.call_deepseek_chat(prompt, use_json_mode=False)

        if res in document_types:
            return res

        # Fuzzy match with threshold
        match, score = process.extractOne(res, document_types)
        
        # If confidence is high enough (80%+), return the match
        if score >= 80:
            return match
        
        # If still no good match, return unknown
        return "unknown"

    def extract_document_fields(self, formatted_text: str, fields_data: dict):
        """
        Extract specific fields from a document using AI.
        
        Args:
            formatted_text: The document text to extract from
            fields_data: Dict mapping field names to sample values
            
        Returns:
            Dict with extracted field values (None for missing fields)
        """
        field_instructions = []
        for field_name, samples in fields_data.items():
            if samples:
                sample_text = f" (examples: {', '.join(samples[:10])})"
                field_instructions.append(f"- {field_name}: {sample_text}")
            else:
                field_instructions.append(f"- {field_name}")
        
        fields_str = "\n".join(field_instructions)
        
        # IMPORTANT: Prompt must mention "json" to use JSON mode
        prompt = f"""
                    Document text:
                    {formatted_text}
        
            Extract the following fields from this document:

            {fields_str}

            Return your response as a JSON object with exactly these field names. Use null for missing fields."""
        
        try:
            # Call with JSON mode enabled (prompt mentions "JSON")
            response = self.call_deepseek_chat(prompt, max_tokens=2000, use_json_mode=True).strip()
            
            # Parse JSON directly
            result = json.loads(response)
            
            # Validate structure
            if not isinstance(result, dict):
                raise ValueError("Expected JSON object")
            
            # Validate and normalize field names
            validated_result = {}
            for field_name in fields_data.keys():
                if field_name in result:
                    validated_result[field_name] = result[field_name]
                else:
                    # Try case-insensitive match
                    found = False
                    for key in result.keys():
                        if key.lower() == field_name.lower():
                            validated_result[field_name] = result[key]
                            found = True
                            break
                    if not found:
                        validated_result[field_name] = None
            
            return validated_result
            
        except json.JSONDecodeError as e:
            print(f"Failed to parse JSON response: {e}")
            print(f"Response was: {response}")
            # Return None for all fields on parse error
            return {field_name: None for field_name in fields_data.keys()}
        
        except Exception as e:
            print(f"Error extracting fields: {e}")
            return {field_name: None for field_name in fields_data.keys()}
    
    def log_token_usage(self, job_id: str, response):
        """Log token usage and cost to database"""
        usage = response.usage
        model = response.model
        
        # Calculate costs
        input_cost = usage.prompt_tokens * self.input_token_price
        output_cost = usage.completion_tokens * self.output_token_price
        
        # Handle cache tokens
        cache_hit_tokens = getattr(usage, 'prompt_cache_hit_tokens', 0) or 0
        cache_creation_tokens = getattr(usage, 'prompt_cache_creation_tokens', 0) or 0
        cache_savings = cache_hit_tokens * (self.input_token_price - self.cache_hit_price)
        
        total_cost = input_cost + output_cost
        
        try:
            self.supabase_client.table("llm_calls").insert({
                "job_id": job_id,
                "organization_id": self.organization_id,
                "model": model,
                "input_tokens": usage.prompt_tokens,
                "output_tokens": usage.completion_tokens,
                "cache_hit_tokens": cache_hit_tokens,
                "cache_creation_tokens": cache_creation_tokens,
                "cost": float(total_cost),
                "cache_savings": float(cache_savings),
                "timestamp": datetime.utcnow().isoformat()
            }).execute()
        except Exception as e:
            print(f"Failed to log LLM usage: {e}")




### PDF Handler

In [53]:
class PDFHandler: 
    def __init__(self, path: str):
        self.path = path
    
    def reconstruct_text_for_llm(self, words_json, y_tolerance=10, space_threshold=15, use_separator=True):
        """
        Reconstructs text from OCR/parsed PDF data in a layout-preserving but LLM-friendly way.
        - Keeps natural layout for textual sections
        - Adds horizontal dividers between vertical gaps
        - Lightly aligns tabular/numeric lines for readability
        """

        # --- Sort words vertically and horizontally ---
        words_sorted = sorted(words_json, key=lambda w: (w["top"], w["x0"]))

        # --- Group words into lines ---
        lines = []
        current_line = []
        last_y = None
        for word in words_sorted:
            if last_y is None or abs(word["top"] - last_y) <= y_tolerance:
                current_line.append(word)
            else:
                lines.append(sorted(current_line, key=lambda w: w["x0"]))
                current_line = [word]
            last_y = word["top"]
        if current_line:
            lines.append(sorted(current_line, key=lambda w: w["x0"]))

        # --- Build each line with spacing ---
        text_lines = []
        prev_line_bottom = None

        for line in lines:
            line_top = min(w["top"] for w in line)
            if prev_line_bottom is not None and (line_top - prev_line_bottom) > (2 * y_tolerance):
                # Add divider for large vertical gaps
                text_lines.append("\n" + "-" * 80 + "\n")

            line_text = []
            prev_right = None

            for word in line:
                if prev_right is not None:
                    gap = word["x0"] - prev_right
                    if gap > space_threshold:
                        if use_separator and gap > 2 * space_threshold:
                            line_text.append(" | ")
                        else:
                            line_text.append("  ")
                    else:
                        line_text.append(" ")
                line_text.append(word["text"])
                prev_right = word.get("x1", word["x0"] + 10)

            text_lines.append("".join(line_text))
            prev_line_bottom = max(w["top"] for w in line)

        # --- Light normalization pass ---
        def clean_layout_for_llm(lines):
            cleaned = []
            for line in lines:
                # Collapse multiple spaces
                line = " ".join(line.split())

                # Normalize separators
                line = line.replace(" |", "|").replace("| ", "|").replace("|", " | ")

                # Detect tabular/numeric lines
                has_numbers = any(ch.isdigit() for ch in line)
                has_pipes = "|" in line
                if has_numbers and (has_pipes or len(line.split()) > 5):
                    # Lightly pad for pseudo-columns
                    parts = [p.strip() for p in line.split("|")]
                    line = " | ".join(p.ljust(25) for p in parts)
                cleaned.append(line)
            return cleaned

        text_lines = clean_layout_for_llm(text_lines)
        return "\n".join(text_lines)

    def extract_pdf_with_layout(self, page_num=0):
        """
        Extract PDF with layout preservation for LLM processing.
        Adds minimal metadata wrapper without assuming document type.
        """
        print(1)
        pdf = pdfplumber.open(self.path)
        print(2)
        words = pdf.pages[page_num].extract_words()
        print(3)
        
        # TODO: Add OCR to words
        
        formatted_text = self.reconstruct_text_for_llm(words)
        
        pdf.close()
        return formatted_text

### Main Script

In [54]:
# --- Supabase Connection ---
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Connected to Supabase")

✅ Connected to Supabase


In [ ]:
# --- Utility: Helper Functions ---
def get_next_pending_job():
    """Fetch one pending or failed job for testing."""
    res = supabase.table("processing_queue") \
        .select("*") \
        .in_("status", ["pending", "failed"]) \
        .order("created_at", desc=False) \
        .limit(1) \
        .execute()
    return res.data[0] if res.data else None

def update_job_status(job_id, status: str):
    """Update the status of a queue job."""
    supabase.table("processing_queue").update({
        "status": status,
        "updated_at": datetime.utcnow().isoformat()
    }).eq("id", job_id).execute()
    print(f"🔁 Job {job_id} → {status}")

def download_file_from_storage(file_path: str, bucket_name: str = "documents"):
    """Download a file from Supabase Storage to a temporary location."""
    try:
        # Download the file from storage
        file_data = supabase.storage.from_(bucket_name).download(file_path)
        
        # Create a temporary file
        suffix = os.path.splitext(file_path)[1]  # Get file extension
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
        temp_file.write(file_data)
        temp_file.close()
        
        return temp_file.name
    except Exception as e:
        print(f"❌ Error downloading file from storage: {e}")
        raise

def get_document_types(organization_id: str):
    doc_types_response = supabase.table("document_types").select("name").eq("organization_id", organization_id).execute()
    return [doc_type["name"] for doc_type in doc_types_response.data] 

def get_schema_definition(organization_id: str, document_type: str):
    schema_definition_response = supabase.table("schema_definition").select("*").eq("organization_id", organization_id).eq("name", document_type).execute()
    return schema_definition_response.data[0]

def insert_document_information(
    extraction_result: dict, 
    job: dict,
    document_type: str,
    supabase_client,
    organization_id: str ):
    """
    Insert extracted data into organization schema tables with status tracking.
    
    Args:
        extraction_result: Result from extract_document_fields
        job: The processing queue job dict containing file info
        document_type: The classified document type
        supabase_client: Supabase client instance
        organization_id: The organization ID
        
    Returns:
        Dict with insertion results including status
    """
    try:
        # Get the organization schema name
        schema_name = f"organization_{organization_id.replace('-', '_')}"
        
        # Extract file name from file path
        file_name = job['file_path'].split('/')[-1]
        
        # Prepare document-level data
        doc_data = extraction_result["document_data"].copy()
        
        # Add required schema fields
        doc_data["id"] = job["document_id"]  # Use the document_id from processing_queue
        doc_data["user_id"] = job["user_id"]  # Required by schema
        doc_data["file_name"] = file_name  # Required by schema
        doc_data["file_path"] = job["file_path"]  # Required by schema
        doc_data["document_type"] = document_type  # What type of document this is
        
        # Map extraction status to document status
        # Schema expects: 'processing', 'completed', 'failed'
        # Our extraction returns: 'complete', 'needs_review'
        if extraction_result["status"] == "complete":
            doc_data["status"] = "completed"
        elif extraction_result["status"] == "needs_review":
            doc_data["status"] = "completed"  # Still completed, just needs review
        else:
            doc_data["status"] = "processing"
        
        # Add metadata fields (these are now in the schema)
        doc_data["needs_manual_review"] = extraction_result["needs_manual_review"]
        doc_data["extraction_metadata"] = extraction_result["extraction_status"]
        
        # Insert into the organization-specific documents table
        doc_response = supabase_client.schema(schema_name).table("documents").upsert(doc_data).execute()
        
        # Insert order lines if present
        order_lines_inserted = 0
        if extraction_result["order_lines"]:
            order_lines_df = extraction_result["order_lines_df"].copy()
            
            # Add document_id to each order line
            order_lines_df["document_id"] = job["document_id"]
            
            # Convert DataFrame to list of dicts for insertion
            order_lines_records = order_lines_df.to_dict('records')
            
            # Insert into the organization-specific order_lines table
            lines_response = supabase_client.schema(schema_name).table("order_lines").insert(order_lines_records).execute()
            order_lines_inserted = len(order_lines_records)
        
        return {
            "success": True,
            "document_inserted": True,
            "order_lines_inserted": order_lines_inserted,
            "status": extraction_result["status"],
            "needs_manual_review": extraction_result["needs_manual_review"],
            "extraction_summary": {
                "extracted_fields": len(extraction_result["extraction_status"]["extracted_fields"]),
                "missing_required_fields": len(extraction_result["extraction_status"]["missing_required_fields"]),
                "order_lines": order_lines_inserted
            }
        }
        
    except Exception as e:
        print(f"Error inserting to Supabase: {e}")
        import traceback
        traceback.print_exc()
        return {
            "success": False,
            "error": str(e),
            "status": "failed",
            "needs_manual_review": True
        }

def process_job(job):
    """Main processing function."""

    job_id = job["id"]
    organization_id = job["organization_id"]

    print(f"⚙️  Processing document: {job['file_path']}")
    update_job_status(job["id"], "processing")

    llm = LLMHandler(supabase, organization_id, job_id)

    # Here you’d call your actual processing pipeline
    try:
        # Download file from Supabase Storage
        print(f"📥 Downloading file from storage...")
        temp_file_path = download_file_from_storage(job["file_path"])
        pdf = PDFHandler(temp_file_path)
        
        # Ingest PDF using PDFPlumber
        print(f"📄 Processing PDF...")
        formatted_text = pdf.extract_pdf_with_layout()
        document_types = get_document_types(job["organization_id"])

        # Classify document
        document_type = llm.classify_document(formatted_text, document_types)

        # Get schema definition
        schema_definition = get_schema_definition(job["organization_id"], document_type)

        # Get document fields
        document_information = llm.extract_document_fields(formatted_text, schema_definition)

        # Insert document information
        insert_document_information(document_information, job["id"], supabase)

        print(f"📄 Document type: {document_type}")
        print(f"📝 Extracted text length: {len(formatted_text)} characters")
        
        # Save the processed document to the database
        print(f"✅ Finished processing: {job['file_path']}")
        update_job_status(job["id"], "completed")

    except Exception as e:
        print(f"❌ Error processing job {job['id']}: {e}")
        update_job_status(job["id"], "failed")
    
    finally:
        # Clean up temporary file
        try:  
            if temp_file_path and os.path.exists(temp_file_path):
                os.remove(temp_file_path)
                print(f"🧹 Cleaned up temporary file")
        except (NameError, Exception) as e:
            # temp_file_path might not be defined if download failed
            pass

def worker_loop():
    """Simple loop to continuously check for new jobs."""
    print("👷 Worker started...")
    while True:
        job = get_next_pending_job()
        if job:
            process_job(job)
        else:
            print("⏸️  No pending jobs. Sleeping...")
            time.sleep(5)

worker_loop()

👷 Worker started...
⚙️  Processing document: organization-ee03aea1-eba1-4024-bda7-01061e199980/unprocessed/1762108658733-mmubmmc34-1da8d132_IN 4500148383 LS0005288(1).PDF
🔁 Job 3a2846fa-ebf5-4399-aac9-c803cc45198f → processing
📥 Downloading file from storage...
📄 Processing PDF...
1
2
3
📄 Document type: Invoice
📝 Extracted text length: 2535 characters
✅ Finished processing: organization-ee03aea1-eba1-4024-bda7-01061e199980/unprocessed/1762108658733-mmubmmc34-1da8d132_IN 4500148383 LS0005288(1).PDF
🔁 Job 3a2846fa-ebf5-4399-aac9-c803cc45198f → completed
🧹 Cleaned up temporary file
⏸️  No pending jobs. Sleeping...
⏸️  No pending jobs. Sleeping...
⏸️  No pending jobs. Sleeping...


KeyboardInterrupt: 

In [3]:
# Test access to tables
# List some tables to check access
tables_to_check = ["users", "processing_queue", "document_types", "organization"]

for table in tables_to_check:
    try:
        res = supabase.table(table).select("*").limit(2).execute()
        print(f"✅ Table '{table}':", res.data)
    except Exception as e:
        print(f"❌ Could not access table '{table}':", e)

✅ Table 'users': [{'id': '415b140b-fe8a-4696-9106-654dfe768adb', 'email': 'thomashteigland@gmail.com', 'first_name': 'Thomas', 'last_name': 'Teigland', 'company': 'Tester', 'phone': '90121618', 'folder_name': 'workspace-91b0e0f6-fa88-4ac1-839d-44413f4c2a49', 'created_at': '2025-10-21T19:47:20.376631+00:00', 'updated_at': '2025-10-21T19:47:21.899383+00:00', 'workspace_id': '91b0e0f6-fa88-4ac1-839d-44413f4c2a49'}]
✅ Table 'processing_queue': [{'id': '54b5fbb3-87ba-4ef6-8f39-1cd55f9b1c28', 'workspace_id': '91b0e0f6-fa88-4ac1-839d-44413f4c2a49', 'document_id': '1761492024029-5d744f90w', 'file_path': 'workspace-91b0e0f6-fa88-4ac1-839d-44413f4c2a49/unprocessed/1761492024029-5d744f90w-Essay - Helsing (2).pdf', 'status': 'completed', 'retry_count': 0, 'error_message': None, 'processing_started_at': None, 'processing_completed_at': None, 'created_at': '2025-10-26T15:20:24.784273+00:00', 'updated_at': '2025-10-26T15:46:39.019729+00:00'}, {'id': '9f45e7ab-bead-41df-9dfd-95b7208600f4', 'workspace_